In [1]:
import os
import re
from pathlib import Path

import pandas as pd

from semevalpolar.finetuning.instruct.predict import generate_predictions_jsonl
from typing import List

from semevalpolar.utils import get_project_root

/home/atharva20240519/polar-semeval-2026


In [2]:
data_path = Path("data/splits/val.jsonl")
with data_path.open("r", encoding="utf-8") as f:
	records = f.readlines()

In [3]:
type(records)

list

In [4]:
import re
import codecs
from typing import List

INPUT_RE = re.compile(
    r'Input:\s*(.*?)\s*Reasoning:',
    re.DOTALL
)

def get_all_inputs(records: List[str]) -> List[str]:
    cleaned: List[str] = []

    for text in records:
        m = INPUT_RE.search(text)
        if not m:
            cleaned.append("")
            continue

        # decode escaped sequences like \\n, \\t
        decoded = codecs.decode(m.group(1), "unicode_escape")

        # normalize all whitespace
        normalized = " ".join(decoded.split())

        cleaned.append(normalized)

    return cleaned


In [5]:
dataset = get_all_inputs(records)

In [6]:
generate_predictions_jsonl(dataset)

Generating predictions for 322 texts...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Generating predictions: 100%|█████████████████████████████████████████████████████████| 322/322 [13:08<00:00,  2.45s/it]


In [10]:
import torch

In [11]:
print(torch.cuda.is_available())

True


In [14]:
import re
import json

def extract_labels_jsonl(input_path, output_path):
    pattern = re.compile(r'Final Answer \(output ONLY 0 or 1\):\s*(\d+)')

    with open(input_path, "r", encoding="utf-8") as fin, \
         open(output_path, "w", encoding="utf-8") as fout:

        for line in fin:
            obj = json.loads(line)

            # Prefer extracted_label if it already exists
            if "extracted_label" in obj:
                label = int(obj["extracted_label"])
            else:
                # Fallback: extract from prediction text
                text = obj.get("prediction", "")
                match = pattern.search(text)
                if match is None:
                    continue  # or raise an error if required
                label = int(match.group(1))

            fout.write(json.dumps({"label": label}) + "\n")

In [15]:
extract_labels_jsonl(os.path.join(get_project_root(), "src", "semevalpolar",
                                  "finetuning", "instruct", "predictions", "predictions.jsonl"),
                     os.path.join(get_project_root(), "src", "semevalpolar",
                                  "finetuning", "instruct", "predictions", "gold.jsonl"))

In [12]:
import json

labels = []
text = []
with open('predictions/predictions.jsonl', 'r') as f:
    for line in f:
        try:
            json_obj = json.loads(line)
            labels.append(json_obj['extracted_label'])
            text.append(json_obj['prediction'])
        except json.JSONDecodeError:
            pass

In [15]:
text[1]

'Input:\nSo The Hill is repeating Russian propaganda too? Propaganda\n\nReasoning:\n1. The text references a news outlet, "The Hill," which implies an organizational affiliation with a specific viewpoint or agenda. 2. The phrase "repeating" suggests a pattern of behavior consistent with the dissemination of information intended to influence public opinion. 3. The use of quotation marks around "propaganda" indicates a deliberate attempt to frame the content as biased rather than neutral.\n\nFinal Answer:\n1'

In [6]:
import json
import re

def extract_final_answers(file_path):
    final_answers = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                json_obj = json.loads(line)
                match = re.search(r'Final Answer:\n(\d+)', json_obj['text'])
                if match:
                    final_answers.append(int(match.group(1)))
            except json.JSONDecodeError:
                pass
    return final_answers

final_answers = extract_final_answers('data/splits/val.jsonl')

In [10]:
comparison = pd.DataFrame({
	"Predictions": labels,
	"Ground Truth": final_answers
})

In [17]:
comparison[comparison["Predictions"] != comparison["Ground Truth"]]

,Predictions,Ground Truth
1,1,0
3,0,1
4,1,0
6,0,1
7,0,1
...,...,...
308,0,1
310,1,0
315,0,1
318,0,1


# 1 Example

In [ ]:
def predict(text):
	prompt = f"""Input:
            {text}

            Reasoning:

            """